# Decay models and diffraction-weighted dose

A voxel that has absorbed a lot of dose diffracts less well. RADDOSE-3D's
decay models say how much less, and the *diffraction-weighted dose* (DWD)
weights each voxel's dose by how much it contributes to the diffraction --
its fluence times its remaining efficiency. So the decay model does not
change the dose, only how the dose is weighted.

In [ ]:
import matplotlib.pyplot as plt

from pyraddose_3d.models.experiment import Experiment
from pyraddose_3d.outputs import image_series
from pyraddose_3d.solve import simulate


def experiment(decay_model, decay_parameters=None):
    """The same crystal, beam and wedge, with a given decay model."""
    return Experiment.model_validate(
        {
            "crystal": {
                "type": "cuboid",
                "dimensions": [40.0, 30.0, 20.0],
                "pixels_per_micron": 0.5,
                "decay_model": decay_model,
                "decay_parameters": decay_parameters,
                "composition": {
                    "method": "unit_cell",
                    "unit_cell": [78.02, 78.02, 78.02],
                    "num_monomers": 24,
                    "num_residues": 51,
                    "protein_heavy_atoms": {"Zn": 0.333, "S": 6},
                    "solvent_heavy_conc": {"P": 425},
                },
            },
            "beam": {
                "type": "gaussian",
                "flux": 2e12,
                "energy": 12.1,
                "fwhm": [15.0, 15.0],
            },
            "wedges": [{"start": 0.0, "end": 180.0, "exposure_time": 60.0}],
        }
    )

## The models

- **simple** (`DDM Simple`): no decay at all, the default.
- **linear** (`DDM Linear`): efficiency falls as 1 - D/86 MGy, reaching
  half at 43 MGy (Owen et al. 2006).
- **bfactor** (`DDM Bfactor`, with `DecayParam` gamma, B0, beta): falls as
  exp(-(D B0 / beta) / (2 gamma^2)); with 2, 3 and 4 it is 1/e at 10.7 MGy.

The Leal et al. (2012) model is not ported yet.

In [ ]:
runs = {
    "simple": simulate(experiment("simple"), record_images=True),
    "linear": simulate(experiment("linear"), record_images=True),
    "bfactor": simulate(experiment("bfactor", (2.0, 3.0, 4.0)), record_images=True),
}
for name, simulation in runs.items():
    summary = simulation.summaries[0]
    print(
        f"{name:8s} max dose {float(summary.max_dose):6.1f} MGy   "
        f"average DWD {float(summary.average_dwd):5.1f} MGy"
    )

The maximum dose is the same for all three; the DWD is not. A decaying
voxel contributes less to the diffraction, and the most-dosed voxels decay
first, so the dose that the *diffraction* sees is lower.

## Image by image

`image_series` gives what `DWDs.csv` and `RDE.csv` hold: the DWD after
each image, and the fluence-weighted mean efficiency of the voxels it
exposed.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True)
for name, simulation in runs.items():
    series = image_series(simulation)
    left.plot(series["angle"], series["dwd"], label=name)
    right.plot(series["angle"], series["weighted"], label=name)
left.set_xlabel("rotation (degrees)")
left.set_ylabel("DWD after the image (MGy)")
right.set_xlabel("rotation (degrees)")
right.set_ylabel("relative diffraction efficiency")
right.legend()
plt.show()

## Resolution-dependent efficiency

Independently of the decay model, `DWDs.csv` estimates how much each
resolution shell has faded since the start, as exp(-DWD / D_e): intensity
at resolution d falls by 1/e every D_e = 81.3 / q^1.7 MGy, with
q = 2 pi / d. High-resolution data fade first:

In [ ]:
series = image_series(runs["simple"])
fig, ax = plt.subplots(figsize=(5.5, 3.8), constrained_layout=True)
for column, label in ((1, "1 A"), (2, "2 A"), (3, "3 A"), (4, "4 A")):
    ax.plot(series["angle"], series["rde"][:, column], label=label)
ax.set_xlabel("rotation (degrees)")
ax.set_ylabel("relative efficiency at resolution")
ax.legend(title="shell")
plt.show()